# LatentDriver Assets Setup (Colab)

This notebook prepares the **evaluation-only** substrate for `LatentDriver`, `PlanT`, and `EasyChauffeur-PPO` on Colab with Drive-backed persistence.

## What this notebook does

1. sync this repo from GitHub
2. mount Google Drive and bind canonical asset directories
3. clone and patch the pinned LatentDriver fork
4. download the public evaluation checkpoints
5. verify that the expected checkpoint files are present

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/achyutmorang/latentdriver-waymax-experiments.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/latentdriver-waymax-experiments")


def run(cmd: list[str], cwd: Path | None = None, env: dict[str, str] | None = None, stream: bool = True) -> str:
    printable = " ".join(str(part) for part in cmd)
    print(f"$ {printable}")
    if stream:
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd) if cwd else None,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        lines: list[str] = []
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            lines.append(line)
        code = proc.wait()
        output = "".join(lines)
        if code != 0:
            raise RuntimeError(f"Command failed with exit code {code}: {printable}")
        return output
    completed = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    return completed.stdout


if REPO_DIR.exists():
    run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
    run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH])
    run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])
else:
    run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)])
repo_rev = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print(json.dumps({"repo_dir": str(REPO_DIR), "repo_rev": repo_rev}, indent=2, sort_keys=True))


In [ ]:
from __future__ import annotations

import json
import os
import sys

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/waymax_research"
os.environ.pop("LATENTDRIVER_RESULTS_ROOT", None)
sys.path.insert(0, str(REPO_DIR / "src"))
from latentdriver_waymax_experiments.colab import bind_drive_layout

binding = bind_drive_layout(DRIVE_ROOT)
print(json.dumps(binding, indent=2, sort_keys=True))


In [ ]:
from __future__ import annotations

import json

run([sys.executable, "scripts/bootstrap_upstream.py"], cwd=REPO_DIR)
lock_path = REPO_DIR / "artifacts" / "locks" / "upstream_bootstrap.json"
print(lock_path.read_text())


In [ ]:
DOWNLOAD_EVALUATION_CHECKPOINTS = True

if DOWNLOAD_EVALUATION_CHECKPOINTS:
    run([sys.executable, "scripts/download_checkpoints.py", "--evaluation-only"], cwd=REPO_DIR)
else:
    print("Skipping checkpoint download.")


In [ ]:
import json

config = json.loads((REPO_DIR / "configs" / "baselines" / "latentdriver_waymax_eval.json").read_text())
checkpoints_root = REPO_DIR / config["assets"]["checkpoints_root"]
summary = {}
for name, spec in config["checkpoints"].items():
    if not spec["method"]:
        continue
    path = checkpoints_root / spec["filename"]
    summary[name] = {
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else None,
        "expected_size_bytes": spec["size_bytes"],
    }
print(json.dumps(summary, indent=2, sort_keys=True))


## Next step

After this notebook succeeds, run [`latentdriver_preprocess_val_colab.ipynb`](./latentdriver_preprocess_val_colab.ipynb) to prepare smoke or full validation artifacts.